# Audio Segmentation Pipeline

This Colab provides a streamlined pipeline for extracting labeled audio segments from Google Cloud Storage (GCS) using JSONL annotations and exporting them back to GCS.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/common/chirp_and_gemini_segment_audio.ipynb)

### Core Functions
1. **Manifest Retrieval**: Fetches target file lists from GCS. Only annotated files are processed.
2. **Local Audio Caching**: Downloads source audio into a local cache. On subsequent runs, it skips downloads if the file is already present locally.
3. **Ground Truth Slicing**: Extract audio segments as defined by the start/end timestamps in the manifest. Optionally adds silence to start and end of audio.
4. **Automated Export**:
    * Uploads processed FLAC segments to GCS organized by `example_id`.
    * Generates and uploads a `batch_manifest.jsonl` containing metadata for all segments (GCS paths, offsets, and durations) to facilitate downstream ASR evaluation.

In [ ]:
# @title Install dependencies
%pip install -q --upgrade \
    loguru \
    soundfile \
    ffmpeg-python

In [ ]:
# @title Imports and environment configuration
import json
import sys
from pathlib import Path
from urllib.parse import urlparse

import ffmpeg
from loguru import logger
from google.colab import userdata

# Load GCP configuration from Colab Secrets
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

# @markdown ### 1. Manifest and Output Configuration
# @markdown The fully-qualified GCS URI to the input manifest file (e.g., gs://wd-transcription-data/manifests/echo/eval/manifest.json)
SOURCE_MANIFEST_URI = "gs://wd-transcription-data/manifests/echo/eval/manifest.json"  # @param {type:"string"}

# @markdown The GCS output path relative to your bucket where the segments should be stored (e.g., segmented_audio/echo/eval)
GCS_OUTPUT_PREFIX = "segmented_audio/echo/eval"  # @param {type:"string"}

# @markdown ### 2. Processing parameters
# @markdown Enable silence padding (1s pre, 2s post)?
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown If False, skip segments that already exist in GCS:
OVERWRITE_EXISTING = False  # @param {type:"boolean"}

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be set in your Colab Secrets."
assert GCS_BUCKET, "GCS_BUCKET must be set in your Colab Secrets."
assert SOURCE_MANIFEST_URI, "SOURCE_MANIFEST_URI must be provided."
assert GCS_OUTPUT_PREFIX, "GCS_OUTPUT_PREFIX must be provided."

LOCAL_BASE_PATH = "/content"
CACHE_DIR = f"{LOCAL_BASE_PATH}/raw_audio"
SEGMENTS_DIR = f"{LOCAL_BASE_PATH}/segments"
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
from google.colab import auth
from google.cloud import storage

# @title Authentication and client initialization
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

# Initialize GCS Client
gcs_client = storage.Client(project=GCP_PROJECT_ID)

In [ ]:
# @title Helper functions
def ensure_local_gcs_audio(gcs_uri: str) -> str:
    """Ensures the audio file from GCS is available locally in CACHE_DIR."""
    parsed = urlparse(gcs_uri)
    bucket_name = parsed.netloc
    blob_name = parsed.path.lstrip("/")

    filename = Path(blob_name).name
    local_path = Path(CACHE_DIR) / filename

    if not local_path.exists():
        logger.info(f"Downloading {filename} from GCS...")
        bucket = gcs_client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(str(local_path))
    return str(local_path)


def cleanup_gcs_output(output_prefix: str) -> None:
    """Deletes existing blobs in the output directory for a clean run."""
    bucket = gcs_client.bucket(GCS_BUCKET)
    blobs = list(bucket.list_blobs(prefix=output_prefix))
    if blobs:
        logger.info(
            f"Cleaning existing files from gs://{GCS_BUCKET}/{output_prefix}..."
        )
        bucket.delete_blobs(blobs)


def run_segmentation_pipeline() -> None:
    """Orchestrates extraction, preserving native audio specs. Fails fast on errors."""
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
    Path(SEGMENTS_DIR).mkdir(parents=True, exist_ok=True)

    # 1. Determine Output Prefix
    # If preprocessing is OFF, we label as _raw. Otherwise, use base path.
    current_output_prefix = (
        GCS_OUTPUT_PREFIX if AUDIO_PREPROCESSING else f"{GCS_OUTPUT_PREFIX}_raw"
    )

    if OVERWRITE_EXISTING:
        cleanup_gcs_output(current_output_prefix)

    # 2. Load the JSON or JSONL manifest from GCS
    parsed_manifest = urlparse(SOURCE_MANIFEST_URI)
    m_bucket = gcs_client.bucket(parsed_manifest.netloc)
    m_blob = m_bucket.blob(parsed_manifest.path.lstrip("/"))
    content = m_blob.download_as_text()

    manifest_data = []
    manifest_file_extension = Path(parsed_manifest.path).suffix.lower()

    if manifest_file_extension == ".json":
        manifest_data = json.loads(content)
        logger.info(f"Loaded JSON manifest with {len(manifest_data)} entries.")
    elif manifest_file_extension == ".jsonl":
        manifest_data = [
            json.loads(line) for line in content.strip().split("\n")
        ]
        logger.info(f"Loaded JSONL manifest with {len(manifest_data)} entries.")
    else:
        raise ValueError(
            f"Unsupported manifest file type: {manifest_file_extension}. Must be .json or .jsonl"
        )

    files_to_process = {}
    for entry in manifest_data:
        path = entry["audio_filepath"]
        if path not in files_to_process:
            files_to_process[path] = []
        files_to_process[path].append(entry)

    output_bucket = gcs_client.bucket(GCS_BUCKET)
    final_manifest_entries = []

    # 3. Process segments
    for gcs_audio_path, segments in files_to_process.items():
        local_src_path = None  # Lazy load only if needed
        example_id = Path(gcs_audio_path).stem

        for i, seg in enumerate(segments):
            seg_id = f"{i:03d}"

            # Explicitly filter by category == "TRANSCRIPTION"
            category = seg.get("category", "").strip().upper()
            if category != "TRANSCRIPTION":
                logger.info(
                    f"Skipping segment {i} because category is {category} (only TRANSCRIPTION is processed)"
                )
                continue

            transcription = seg.get("text", "").strip()
            start_s = seg["offset"]
            duration_s = seg["duration"]
            filename = f"{example_id}__seg{seg_id}.flac"
            blob_name = f"{current_output_prefix}/{example_id}/{filename}"
            output_blob = output_bucket.blob(blob_name)

            total_duration = (
                duration_s + 3 if AUDIO_PREPROCESSING else duration_s
            )

            # Skip if file exists and we aren't overwriting
            if not OVERWRITE_EXISTING and output_blob.exists():
                final_manifest_entries.append(
                    {
                        "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                        "example_id": example_id,
                        "original_offset": start_s,
                        "duration": total_duration,
                        "segment_id": seg_id,
                        "text": transcription,
                    }
                )
                continue

            # Lazy load source audio and probe specs (fidelity preserved)
            if local_src_path is None:
                local_src_path = ensure_local_gcs_audio(gcs_audio_path)
                probe = ffmpeg.probe(local_src_path)
                audio_stream = next(
                    (s for s in probe["streams"] if s["codec_type"] == "audio"),
                    None,
                )
                if not audio_stream:
                    raise ValueError(f"No audio found in {gcs_audio_path}")
                native_sr = int(audio_stream["sample_rate"])
                native_channels = int(audio_stream["channels"])
                logger.info(
                    f"Processing {example_id} ({native_sr}Hz, {native_channels}ch)"
                )

            # Per-segment progress logging
            logger.info(
                f"  > Slicing segment {seg_id} (offset: {start_s}s, duration: {duration_s}s)"
            )

            local_slice_path = Path(SEGMENTS_DIR) / filename

            # Build FFmpeg chain
            input_stream = ffmpeg.input(
                local_src_path, ss=start_s, t=duration_s
            )

            if AUDIO_PREPROCESSING:
                # Generate silence matching native channel layout
                layout = "mono" if native_channels == 1 else "stereo"
                silence_pre = ffmpeg.input(
                    f"anullsrc=r={native_sr}:cl={layout}", f="lavfi", t=1
                )
                silence_post = ffmpeg.input(
                    f"anullsrc=r={native_sr}:cl={layout}", f="lavfi", t=2
                )
                processed_stream = ffmpeg.concat(
                    silence_pre, input_stream, silence_post, v=0, a=1
                )
            else:
                processed_stream = input_stream

            # Final output encoding
            out = ffmpeg.output(processed_stream, str(local_slice_path))
            ffmpeg.run(out, overwrite_output=True, quiet=True)

            # Upload to GCS
            output_blob.upload_from_filename(str(local_slice_path))
            # Clean up local slice file after upload
            local_slice_path.unlink()

            final_manifest_entries.append(
                {
                    "audio_filepath": f"gs://{GCS_BUCKET}/{blob_name}",
                    "example_id": example_id,
                    "original_offset": start_s,
                    "duration": total_duration,
                    "segment_id": seg_id,
                    "text": transcription,
                }
            )

        # Clean up local source audio file after all its segments are processed
        if local_src_path and Path(local_src_path).exists():
            Path(local_src_path).unlink()

    # 4. Final Manifest
    local_manifest = Path(LOCAL_BASE_PATH) / BATCH_MANIFEST_FILENAME
    with open(local_manifest, "w") as f:
        for entry in final_manifest_entries:
            f.write(json.dumps(entry) + "\n")

    output_bucket.blob(
        f"{current_output_prefix}/{BATCH_MANIFEST_FILENAME}"
    ).upload_from_filename(str(local_manifest))
    logger.info(
        f"Done. {len(final_manifest_entries)} entries in manifest for {current_output_prefix}."
    )

In [ ]:
# @title Create the segments and manifest file
run_segmentation_pipeline()